# 🏥 HealthBridge Medical Center
## Notebook 02 — Exploratory Data Analysis (EDA)
**ADTA 5340 | Advanced Data Analytics Capstone | AY 2025–2026**

---

### What this notebook covers
This EDA is split into **two phases**:

**Phase A — Clean Data EDA** (from `diabetic_clean.xlsx`)
Analyzes the raw patient characteristics *before* encoding — uses the original readable labels like age groups, race names, and ICD-9 codes. Best for understanding the population and telling the clinical story.

**Phase B — Preprocessed Data EDA** (from `X_train.xlsx` / `X_test.xlsx`)
Analyzes the model-ready encoded feature matrix *after* preprocessing. Used to validate the pipeline, understand feature distributions after SMOTE, and identify the most correlated predictors.

---
### Files needed
| File | Source |
|---|---|
| `diabetic_clean.xlsx` | Output of Notebook 01 — Data Cleaning |
| `X_train.xlsx` | Output of Notebook 03 — Preprocessing |
| `X_test.xlsx` | Output of Notebook 03 — Preprocessing |
| `y_train.xlsx` | Output of Notebook 03 — Preprocessing |
| `y_test.xlsx` | Output of Notebook 03 — Preprocessing |

## CELL 1 — Install & Import Libraries

**What:** Load every Python library needed for this notebook.

**Why:** Pandas handles tables, Matplotlib and Seaborn create charts, NumPy handles math, and warnings suppression keeps the output clean.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Display settings ──────────────────────────────────────────────────────────
pd.set_option('display.max_columns', 80)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_style('whitegrid')
sns.set_palette('muted')
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

# ── Color palette (consistent throughout notebook) ────────────────────────────
C_BLUE   = '#2E75B6'   # Not readmitted / general bars
C_RED    = '#C00000'   # Readmitted <30 days / risk highlight
C_GREEN  = '#1D7A4F'   # Positive outcome
C_AMBER  = '#D4860A'   # Warning / medium risk
C_GRAY   = '#888780'   # Neutral

print('✅ Libraries loaded successfully')

## CELL 2 — Load All Data Files

**What:** Read all five files into Python DataFrames.

**Why:** We need the clean file for clinical EDA and the preprocessed files for model-feature EDA. Loading them all upfront confirms the pipeline ran correctly.

> Update the file paths if your files are in Google Drive.

In [ ]:
# ── Option A: Files uploaded to Colab session ─────────────────────────────────
df       = pd.read_excel('diabetic_clean.xlsx', engine='openpyxl')
X_train  = pd.read_excel('X_train.xlsx',        engine='openpyxl')
X_test   = pd.read_excel('X_test.xlsx',         engine='openpyxl')
y_train  = pd.read_excel('y_train.xlsx',        engine='openpyxl').squeeze()
y_test   = pd.read_excel('y_test.xlsx',         engine='openpyxl').squeeze()

# ── Option B: Files in Google Drive (uncomment if needed) ─────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# FOLDER = '/content/drive/MyDrive/HealthBridge_Project'
# df      = pd.read_excel(f'{FOLDER}/diabetic_clean.xlsx', engine='openpyxl')
# X_train = pd.read_excel(f'{FOLDER}/X_train.xlsx',        engine='openpyxl')
# X_test  = pd.read_excel(f'{FOLDER}/X_test.xlsx',         engine='openpyxl')
# y_train = pd.read_excel(f'{FOLDER}/y_train.xlsx',        engine='openpyxl').squeeze()
# y_test  = pd.read_excel(f'{FOLDER}/y_test.xlsx',         engine='openpyxl').squeeze()

# ── Create binary target on clean file ───────────────────────────────────────
df['readmitted_30'] = (df['readmitted'] == '<30').astype(int)

# ── Summary report ────────────────────────────────────────────────────────────
print('=' * 55)
print('  FILES LOADED — SUMMARY')
print('=' * 55)
print(f'  diabetic_clean.xlsx  →  {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'  X_train.xlsx         →  {X_train.shape[0]:,} rows × {X_train.shape[1]} features')
print(f'  X_test.xlsx          →  {X_test.shape[0]:,} rows × {X_test.shape[1]} features')
print(f'  y_train positive:    {y_train.sum():,} / {len(y_train):,}  ({y_train.mean()*100:.1f}%)')
print(f'  y_test positive:     {y_test.sum():,}  / {len(y_test):,}  ({y_test.mean()*100:.1f}%)')
print('=' * 55)

---
# PHASE A — Clean Data EDA
### *Using diabetic_clean.xlsx — original readable labels*
---

## CELL 3 — Target Variable Distribution

**What:** Visualize how patients are distributed across the three readmission categories, then show the final binary target.

**Why:** This is always the first chart in any ML project — you need to see the class imbalance clearly before making any modeling decisions.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# ── Chart 1: 3-class distribution (bar) ──────────────────────────────────────
vc = df['readmitted'].value_counts()
colors_3 = [C_GREEN, C_BLUE, C_RED]
bars = axes[0].bar(vc.index, vc.values, color=colors_3, edgecolor='black', linewidth=0.5)
axes[0].set_title('Original: 3-Class Readmission')
axes[0].set_xlabel('Readmitted')
axes[0].set_ylabel('Patient Count')
for bar, val in zip(bars, vc.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 300,
                 f'{val:,}\n({val/len(df)*100:.1f}%)', ha='center', fontsize=9, fontweight='bold')

# ── Chart 2: Binary target (bar) ─────────────────────────────────────────────
bc = df['readmitted_30'].value_counts().sort_index()
labels = ['Not Readmitted\nwithin 30 days (0)', 'Readmitted\nwithin 30 days (1)']
bars2 = axes[1].bar(labels, bc.values, color=[C_BLUE, C_RED], edgecolor='black', linewidth=0.5)
axes[1].set_title('Binary Target: readmitted_30')
axes[1].set_ylabel('Patient Count')
for bar, val in zip(bars2, bc.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 300,
                 f'{val:,}\n({val/len(df)*100:.1f}%)', ha='center', fontsize=9, fontweight='bold')

# ── Chart 3: Pie chart ────────────────────────────────────────────────────────
axes[2].pie(bc.values, labels=['Not Readmitted\n<30 days (91%)', 'Readmitted\n<30 days (9%)'],
            colors=[C_BLUE, C_RED], autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 10}, explode=(0, 0.08))
axes[2].set_title('Class Imbalance — Binary Target')

plt.suptitle('HealthBridge — Target Variable Analysis (n=69,990)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('eda_01_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Key finding: Only {df["readmitted_30"].mean()*100:.1f}% of patients are readmitted within 30 days')
print(f'Class imbalance ratio: {(df["readmitted_30"]==0).sum() / (df["readmitted_30"]==1).sum():.1f}:1 (negative:positive)')

## CELL 4 — Patient Demographics Overview

**What:** Visualize the distribution of age, gender, and race across the dataset.

**Why:** Understanding who your patient population is matters for both the model and the business case. HealthBridge's patient base is predominantly elderly — important context for the HRRP.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Age distribution ──────────────────────────────────────────────────────────
age_order = ['[0-10)','[10-20)','[20-30)','[30-40)','[40-50)',
             '[50-60)','[60-70)','[70-80)','[80-90)','[90-100)']
age_counts = df['age'].value_counts().reindex(age_order)
bars = axes[0].bar(age_order, age_counts.values, color=C_BLUE, edgecolor='black', linewidth=0.5)
axes[0].set_title('Patient Count by Age Group')
axes[0].set_xlabel('Age Group')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)
# Highlight the peak age group
peak_idx = age_counts.values.argmax()
bars[peak_idx].set_color(C_RED)
bars[peak_idx].set_edgecolor('black')

# ── Gender distribution ───────────────────────────────────────────────────────
gender_counts = df[df['gender'] != 'Unknown/Invalid']['gender'].value_counts()
axes[1].pie(gender_counts.values, labels=gender_counts.index,
            colors=[C_BLUE, C_AMBER], autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 11})
axes[1].set_title('Gender Distribution')

# ── Race distribution ─────────────────────────────────────────────────────────
race_counts = df[df['race'] != 'Unknown']['race'].value_counts()
colors_race = [C_BLUE, C_RED, C_GREEN, C_AMBER, C_GRAY]
bars_r = axes[2].barh(race_counts.index, race_counts.values,
                       color=colors_race[:len(race_counts)], edgecolor='black', linewidth=0.5)
axes[2].set_title('Race Distribution')
axes[2].set_xlabel('Patient Count')
for bar, val in zip(bars_r, race_counts.values):
    axes[2].text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
                 f'{val:,} ({val/race_counts.sum()*100:.1f}%)', va='center', fontsize=9)

plt.suptitle('HealthBridge Patient Demographics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_02_demographics.png', dpi=150, bbox_inches='tight')
plt.show()

over60 = df[df['age'].isin(['[60-70)','[70-80)','[80-90)','[90-100)'])].shape[0]
print(f'Patients aged 60+: {over60:,} ({over60/len(df)*100:.1f}%) — primary HRRP risk group')

## CELL 5 — Readmission Rate by Age Group

**What:** Calculate and plot the 30-day readmission *rate* (%) for each age group.

**Why:** Seeing patient counts (Cell 4) tells you who is in the hospital. Seeing readmission *rates* tells you who is most likely to come back — these are very different questions and this chart answers the one that matters for the model.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Readmission rate by age
age_readmit = df.groupby('age')['readmitted_30'].agg(['mean','count']).reindex(age_order)
age_readmit['rate_pct'] = age_readmit['mean'] * 100
overall_rate = df['readmitted_30'].mean() * 100

bar_colors = [C_RED if r > overall_rate else C_BLUE for r in age_readmit['rate_pct']]
bars = axes[0].bar(age_order, age_readmit['rate_pct'], color=bar_colors,
                   edgecolor='black', linewidth=0.5)
axes[0].axhline(overall_rate, color='navy', linestyle='--', linewidth=1.5,
                label=f'Overall avg: {overall_rate:.1f}%')
axes[0].set_title('30-Day Readmission Rate by Age Group')
axes[0].set_xlabel('Age Group')
axes[0].set_ylabel('Readmission Rate (%)')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend()
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
for bar, rate in zip(bars, age_readmit['rate_pct']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{rate:.1f}%', ha='center', fontsize=8)

# Readmission rate by gender
gender_readmit = df[df['gender'] != 'Unknown/Invalid'].groupby('gender')['readmitted_30'].agg(['mean','count'])
gender_readmit['rate_pct'] = gender_readmit['mean'] * 100
bars_g = axes[1].bar(gender_readmit.index, gender_readmit['rate_pct'],
                     color=[C_BLUE, C_AMBER], edgecolor='black', linewidth=0.5, width=0.4)
axes[1].axhline(overall_rate, color='navy', linestyle='--', linewidth=1.5,
                label=f'Overall avg: {overall_rate:.1f}%')
axes[1].set_title('30-Day Readmission Rate by Gender')
axes[1].set_ylabel('Readmission Rate (%)')
axes[1].legend()
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
for bar, (_, row) in zip(bars_g, gender_readmit.iterrows()):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{row["rate_pct"]:.1f}%\n(n={int(row["count"]):,})', ha='center', fontsize=10)

plt.suptitle('Readmission Rates by Demographics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_03_readmission_by_demographics.png', dpi=150, bbox_inches='tight')
plt.show()

print('Age group insights:')
for age, row in age_readmit.iterrows():
    flag = ' ← ABOVE AVERAGE' if row['rate_pct'] > overall_rate else ''
    print(f'  {age:<12} {row["rate_pct"]:5.1f}%  (n={int(row["count"]):,}){flag}')

## CELL 6 — Prior Inpatient Visits vs Readmission Rate

**What:** Plot readmission rates by how many prior inpatient hospitalizations a patient had in the past year.

**Why:** `number_inpatient` is the strongest individual predictor of readmission in this dataset. This chart makes that relationship visually undeniable — a core finding for your presentation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cap at 6+ for readability
df['inpatient_capped'] = df['number_inpatient'].clip(upper=6)
label_map = {0:'0',1:'1',2:'2',3:'3',4:'4',5:'5',6:'6+'}

# Chart 1: Readmission rate by prior visits
in_readmit = df.groupby('inpatient_capped')['readmitted_30'].agg(['mean','count'])
in_readmit['rate_pct'] = in_readmit['mean'] * 100
in_readmit.index = [label_map[i] for i in in_readmit.index]

bar_colors = [C_GREEN if r <= overall_rate else C_AMBER if r <= 20 else C_RED
              for r in in_readmit['rate_pct']]
bars = axes[0].bar(in_readmit.index, in_readmit['rate_pct'],
                   color=bar_colors, edgecolor='black', linewidth=0.5)
axes[0].axhline(overall_rate, color='navy', linestyle='--', linewidth=1.5,
                label=f'Overall avg: {overall_rate:.1f}%')
axes[0].set_title('Readmission Rate by Prior Inpatient Visits')
axes[0].set_xlabel('Prior Inpatient Visits (Past Year)')
axes[0].set_ylabel('Readmission Rate (%)')
axes[0].legend()
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
for bar, (idx, row) in zip(bars, in_readmit.iterrows()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{row["rate_pct"]:.1f}%', ha='center', fontsize=9, fontweight='bold')

# Chart 2: Distribution of prior visits
count_series = in_readmit['count']
axes[1].bar(count_series.index, count_series.values, color=C_BLUE, edgecolor='black', linewidth=0.5)
axes[1].set_title('Patient Count by Prior Inpatient Visits')
axes[1].set_xlabel('Prior Inpatient Visits (Past Year)')
axes[1].set_ylabel('Patient Count')
for i, (idx, val) in enumerate(count_series.items()):
    axes[1].text(i, val + 200, f'{val:,}', ha='center', fontsize=9)

plt.suptitle('Prior Hospitalization History vs 30-Day Readmission Risk', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_04_prior_visits.png', dpi=150, bbox_inches='tight')
plt.show()

print('Key finding — Prior inpatient visits is the strongest predictor:')
for idx, row in in_readmit.iterrows():
    print(f'  {idx} prior visits → {row["rate_pct"]:.1f}% readmission rate  (n={int(row["count"]):,})')

## CELL 7 — Numeric Feature Distributions

**What:** Plot the distribution (histogram) of every key numeric feature, split by readmission outcome.

**Why:** Comparing distributions between readmitted and non-readmitted patients reveals where the two groups differ — those differences are what the model learns from.

In [ ]:
numeric_features = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'number_diagnoses'
]

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

readmitted_group     = df[df['readmitted_30'] == 1]
not_readmitted_group = df[df['readmitted_30'] == 0]

for i, col in enumerate(numeric_features):
    axes[i].hist(not_readmitted_group[col], bins=30, alpha=0.6,
                 color=C_BLUE, label='Not Readmitted (0)', edgecolor='white', density=True)
    axes[i].hist(readmitted_group[col], bins=30, alpha=0.6,
                 color=C_RED, label='Readmitted <30d (1)', edgecolor='white', density=True)
    axes[i].set_title(col.replace('_', ' ').title())
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Density')
    # Add mean lines
    axes[i].axvline(not_readmitted_group[col].mean(), color=C_BLUE,
                    linestyle='--', linewidth=1.5, label=f'Mean (0): {not_readmitted_group[col].mean():.1f}')
    axes[i].axvline(readmitted_group[col].mean(), color=C_RED,
                    linestyle='--', linewidth=1.5, label=f'Mean (1): {readmitted_group[col].mean():.1f}')
    if i == 0:
        axes[i].legend(fontsize=8)

plt.suptitle('Feature Distributions: Readmitted vs Not Readmitted', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_05_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

# Mean comparison table
print('\nMean comparison — Readmitted vs Not Readmitted:')
print(f'{"Feature":<25} {"Not Readmitted":>16} {"Readmitted <30d":>16} {"Difference":>12}')
print('-' * 72)
for col in numeric_features:
    m0 = not_readmitted_group[col].mean()
    m1 = readmitted_group[col].mean()
    diff = m1 - m0
    flag = ' ↑' if diff > 0 else ' ↓'
    print(f'{col:<25} {m0:>16.2f} {m1:>16.2f} {diff:>+11.2f}{flag}')

## CELL 8 — Readmission Rate by Diagnosis Category

**What:** Show readmission rates for each ICD-9 clinical category (Circulatory, Respiratory, Diabetes, etc.) from the primary diagnosis column.

**Why:** Directly maps to the HRRP-qualifying conditions. Circulatory (Heart Failure, AMI) and Respiratory (COPD, Pneumonia) are the two categories that drive HealthBridge's penalty exposure.

In [ ]:
# ICD-9 grouping function (same as used in preprocessing)
def map_icd9(code):
    if pd.isna(code) or code == 'Unknown': return 'Other'
    code = str(code)
    if code.startswith('V') or code.startswith('E'): return 'Other'
    try:
        n = float(code.split('.')[0])
        if 390 <= n <= 459: return 'Circulatory'      # HF (428), AMI (410)
        if 460 <= n <= 519: return 'Respiratory'      # COPD (491-496), Pneumonia (486)
        if n == 250:        return 'Diabetes'
        if 240 <= n <= 279: return 'Endocrine'
        if 800 <= n <= 999: return 'Injury'
        if 710 <= n <= 739: return 'Musculoskeletal'  # Hip/Knee
        if 140 <= n <= 239: return 'Neoplasms'
        if 520 <= n <= 579: return 'Digestive'
        return 'Other'
    except: return 'Other'

df['diag_1_cat'] = df['diag_1'].apply(map_icd9)

diag_readmit = df.groupby('diag_1_cat')['readmitted_30'].agg(['mean','count'])
diag_readmit['rate_pct'] = diag_readmit['mean'] * 100
diag_readmit = diag_readmit[diag_readmit['count'] > 200].sort_values('rate_pct', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Chart 1: Readmission rate by diagnosis category
bar_colors_d = [C_RED if r > overall_rate else C_BLUE for r in diag_readmit['rate_pct']]
bars = axes[0].barh(diag_readmit.index, diag_readmit['rate_pct'],
                    color=bar_colors_d, edgecolor='black', linewidth=0.5)
axes[0].axvline(overall_rate, color='navy', linestyle='--', linewidth=1.5,
                label=f'Overall avg: {overall_rate:.1f}%')
axes[0].set_title('Readmission Rate by Primary Diagnosis Category')
axes[0].set_xlabel('30-Day Readmission Rate (%)')
axes[0].legend()
axes[0].xaxis.set_major_formatter(mtick.PercentFormatter())
for bar, (_, row) in zip(bars, diag_readmit.iterrows()):
    axes[0].text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                 f'{row["rate_pct"]:.1f}%  (n={int(row["count"]):,})', va='center', fontsize=9)

# Chart 2: Patient volume by diagnosis category
diag_vol = diag_readmit.sort_values('count', ascending=True)
axes[1].barh(diag_vol.index, diag_vol['count'], color=C_BLUE, edgecolor='black', linewidth=0.5)
axes[1].set_title('Patient Volume by Primary Diagnosis Category')
axes[1].set_xlabel('Patient Count')
for i, (_, row) in enumerate(diag_vol.iterrows()):
    axes[1].text(row['count'] + 50, i, f'{int(row["count"]):,}', va='center', fontsize=9)

plt.suptitle('Primary Diagnosis Category — Volume and Readmission Risk\n(Red = above average risk  |  HRRP conditions: Circulatory, Respiratory, Musculoskeletal)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_06_diagnosis_categories.png', dpi=150, bbox_inches='tight')
plt.show()

## CELL 9 — Medication & Treatment Analysis

**What:** Compare readmission rates for patients on insulin and diabetes medications versus those not on them. Also visualize the A1C test result distribution.

**Why:** This is a diabetes patient cohort — medication management and glycemic control are clinically central. A1Cresult is a key indicator of how well diabetes is controlled, and medication changes at discharge are a strong readmission signal.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Chart 1: Insulin use vs readmission
insulin_readmit = df.groupby('insulin')['readmitted_30'].agg(['mean','count'])
insulin_readmit['rate_pct'] = insulin_readmit['mean'] * 100
bars = axes[0].bar(insulin_readmit.index, insulin_readmit['rate_pct'],
                   color=[C_GREEN, C_BLUE, C_AMBER, C_RED], edgecolor='black', linewidth=0.5)
axes[0].axhline(overall_rate, color='navy', linestyle='--', linewidth=1.5, label='Overall avg')
axes[0].set_title('Readmission Rate by Insulin Status')
axes[0].set_xlabel('Insulin Status')
axes[0].set_ylabel('Readmission Rate (%)')
axes[0].legend(fontsize=9)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
for bar, (idx, row) in zip(bars, insulin_readmit.iterrows()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{row["rate_pct"]:.1f}%\n(n={int(row["count"]):,})', ha='center', fontsize=8)

# Chart 2: DiabetesMed vs readmission
dm_readmit = df.groupby('diabetesMed')['readmitted_30'].agg(['mean','count'])
dm_readmit['rate_pct'] = dm_readmit['mean'] * 100
bars2 = axes[1].bar(dm_readmit.index, dm_readmit['rate_pct'],
                    color=[C_BLUE, C_RED], edgecolor='black', linewidth=0.5, width=0.4)
axes[1].axhline(overall_rate, color='navy', linestyle='--', linewidth=1.5, label='Overall avg')
axes[1].set_title('Readmission Rate by Diabetes Medication')
axes[1].set_xlabel('Diabetes Medication Prescribed')
axes[1].set_ylabel('Readmission Rate (%)')
axes[1].legend(fontsize=9)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
for bar, (idx, row) in zip(bars2, dm_readmit.iterrows()):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{row["rate_pct"]:.1f}%\n(n={int(row["count"]):,})', ha='center', fontsize=9)

# Chart 3: A1C result distribution
a1c_valid = df[df['A1Cresult'].notna() & (df['A1Cresult'] != 'None')]
if len(a1c_valid) > 0:
    a1c_readmit = a1c_valid.groupby('A1Cresult')['readmitted_30'].agg(['mean','count'])
    a1c_readmit['rate_pct'] = a1c_readmit['mean'] * 100
    axes[2].bar(a1c_readmit.index, a1c_readmit['rate_pct'],
                color=C_BLUE, edgecolor='black', linewidth=0.5)
    axes[2].axhline(overall_rate, color='navy', linestyle='--', linewidth=1.5, label='Overall avg')
    axes[2].set_title('Readmission Rate by A1C Result\n(Patients Tested Only)')
    axes[2].set_xlabel('A1C Result')
    axes[2].set_ylabel('Readmission Rate (%)')
    axes[2].legend(fontsize=9)
    axes[2].yaxis.set_major_formatter(mtick.PercentFormatter())
    for i, (idx, row) in enumerate(a1c_readmit.iterrows()):
        axes[2].text(i, row['rate_pct'] + 0.1, f'{row["rate_pct"]:.1f}%', ha='center', fontsize=9)
else:
    axes[2].text(0.5, 0.5, 'A1C data not available\n(filled with None during cleaning)',
                 ha='center', va='center', transform=axes[2].transAxes, fontsize=11)
    axes[2].set_title('A1C Result')

plt.suptitle('Medication & Glycemic Control vs 30-Day Readmission', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_07_medications.png', dpi=150, bbox_inches='tight')
plt.show()

## CELL 10 — Correlation Heatmap

**What:** Calculate and visualize the pairwise correlation between all numeric features and the target variable.

**Why:** Identifies which features move together (multicollinearity risk) and which have the strongest linear relationship with readmission. This informs feature selection and tells you which variables the model is most likely to rely on.

In [ ]:
numeric_cols_heatmap = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'number_diagnoses', 'readmitted_30'
]

corr_matrix = df[numeric_cols_heatmap].corr()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Chart 1: Full correlation heatmap
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, ax=axes[0],
            cbar_kws={'shrink': 0.8})
axes[0].set_title('Correlation Heatmap — Numeric Features')
axes[0].tick_params(axis='x', rotation=45)

# Chart 2: Bar chart of correlation with target only
target_corr = corr_matrix['readmitted_30'].drop('readmitted_30').sort_values()
bar_colors_c = [C_RED if v > 0 else C_BLUE for v in target_corr.values]
axes[1].barh(target_corr.index, target_corr.values, color=bar_colors_c,
             edgecolor='black', linewidth=0.5)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Feature Correlation with readmitted_30\n(Red = positive, Blue = negative)')
axes[1].set_xlabel('Pearson Correlation Coefficient')
for i, (feat, val) in enumerate(target_corr.items()):
    axes[1].text(val + (0.002 if val >= 0 else -0.002), i,
                 f'{val:+.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=9)

plt.suptitle('Feature Correlation Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_08_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop correlations with readmitted_30 (absolute value):')
print(target_corr.abs().sort_values(ascending=False).to_string())

---
# PHASE B — Preprocessed Data EDA
### *Using X_train / X_test — model-ready encoded features*
---

## CELL 11 — SMOTE Class Balance Validation

**What:** Compare the class distribution before and after SMOTE — in training vs test sets.

**Why:** Verifies SMOTE worked correctly. Training should be ~50/50. Test should still be ~9% positive. If these don't match, something went wrong in preprocessing.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

datasets = [
    ('Original Clean Data', df['readmitted_30']),
    ('Training Set (after SMOTE)', y_train),
    ('Test Set (original imbalance)', y_test),
]

for ax, (title, y) in zip(axes, datasets):
    vc = y.value_counts().sort_index()
    bars = ax.bar(['Not Readmitted (0)', 'Readmitted <30d (1)'],
                  vc.values, color=[C_BLUE, C_RED], edgecolor='black', linewidth=0.5)
    ax.set_title(title)
    ax.set_ylabel('Count')
    for bar, val in zip(bars, vc.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vc.values)*0.01,
                f'{val:,}\n({val/len(y)*100:.1f}%)', ha='center', fontsize=9, fontweight='bold')
    ax.set_ylim(0, max(vc.values) * 1.2)
    ax.tick_params(axis='x', rotation=15)

plt.suptitle('Class Balance: Before SMOTE → After SMOTE → Test Set', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_09_smote_validation.png', dpi=150, bbox_inches='tight')
plt.show()

print('SMOTE Validation Report:')
print(f'  Original data positive class:    {df["readmitted_30"].mean()*100:.1f}%')
print(f'  Training set positive class:     {y_train.mean()*100:.1f}%  ← should be ~50%')
print(f'  Test set positive class:         {y_test.mean()*100:.1f}%   ← should be ~9%')
smote_ok = abs(y_train.mean() - 0.5) < 0.05 and abs(y_test.mean() - 0.09) < 0.03
print(f'  SMOTE applied correctly:         {"✅ YES" if smote_ok else "⚠️  CHECK PREPROCESSING"}')

## CELL 12 — Feature Distribution After Scaling

**What:** Plot the distributions of scaled numeric features to confirm StandardScaler was applied correctly.

**Why:** After StandardScaler, every numeric column should be centered near 0 with a standard deviation near 1. If any feature is wildly off-center, it may indicate a scaling error.

In [ ]:
scaled_features = ['time_in_hospital', 'num_lab_procedures', 'num_procedures',
                   'num_medications', 'number_outpatient', 'number_emergency',
                   'number_inpatient', 'number_diagnoses']
scaled_features = [f for f in scaled_features if f in X_train.columns]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(scaled_features):
    axes[i].hist(X_train[col], bins=40, color=C_BLUE, alpha=0.7, edgecolor='white', density=True)
    mean_val = X_train[col].mean()
    std_val  = X_train[col].std()
    axes[i].axvline(0, color='red', linestyle='--', linewidth=1.5, label='0 (expected mean)')
    axes[i].axvline(mean_val, color='orange', linestyle='-', linewidth=1.5,
                    label=f'Actual mean: {mean_val:.2f}')
    axes[i].set_title(f'{col.replace("_"," ").title()}\nMean={mean_val:.2f}, Std={std_val:.2f}')
    axes[i].set_xlabel('Scaled Value')
    axes[i].set_ylabel('Density')
    if i == 0:
        axes[i].legend(fontsize=8)

plt.suptitle('Scaled Numeric Features Distribution (X_train after StandardScaler)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_10_scaled_features.png', dpi=150, bbox_inches='tight')
plt.show()

print('Scaling validation — numeric features in X_train:')
print(f'{"Feature":<25} {"Mean (≈0?)": >14} {"Std (≈1?)": >12} {"Status"}')
print('-' * 65)
for col in scaled_features:
    m, s = X_train[col].mean(), X_train[col].std()
    ok = abs(m) < 0.2 and abs(s - 1) < 0.2
    print(f'{col:<25} {m:>14.4f} {s:>12.4f}   {"✅" if ok else "⚠️ CHECK"}')

## CELL 13 — Top Correlated Features with Target (Preprocessed)

**What:** Recalculate correlation between all 71 encoded features and the target using the preprocessed training data.

**Why:** This shows the full picture after encoding — including the one-hot encoded diagnosis categories, medication flags, and race variables that weren't in the clean file correlation (Cell 10). These correlations are what the model actually sees.

In [ ]:
# Combine X_train with y_train to calculate correlations
train_combined = X_train.copy()
train_combined['target'] = y_train.values

feature_corr = train_combined.corr()['target'].drop('target').abs().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Top 20 features
top20 = feature_corr.head(20)
bar_cols = [C_RED if 'inpatient' in f or 'time' in f or 'diag' in f
            else C_BLUE for f in top20.index]
axes[0].barh(top20.index[::-1], top20.values[::-1],
             color=bar_cols[::-1], edgecolor='black', linewidth=0.5)
axes[0].set_title('Top 20 Features — Absolute Correlation\nwith readmitted_30 (Preprocessed X_train)')
axes[0].set_xlabel('|Pearson Correlation|')
for i, (feat, val) in enumerate(zip(top20.index[::-1], top20.values[::-1])):
    axes[0].text(val + 0.001, i, f'{val:.3f}', va='center', fontsize=9)

# Bottom 20 (near zero correlation — potential drop candidates)
bottom20 = feature_corr.tail(20)
axes[1].barh(bottom20.index[::-1], bottom20.values[::-1],
             color=C_GRAY, edgecolor='black', linewidth=0.5)
axes[1].set_title('Bottom 20 Features — Lowest Correlation\n(Potential low-signal features)')
axes[1].set_xlabel('|Pearson Correlation|')
for i, (feat, val) in enumerate(zip(bottom20.index[::-1], bottom20.values[::-1])):
    axes[1].text(val + 0.0001, i, f'{val:.4f}', va='center', fontsize=9)

plt.suptitle('Feature-Target Correlation Analysis — Model Feature Set (71 Features)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_11_feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 15 most correlated features with readmitted_30:')
for i, (feat, val) in enumerate(feature_corr.head(15).items(), 1):
    print(f'  {i:2}. {feat:<45} {val:.4f}')

## CELL 14 — Feature Group Analysis (Encoded)

**What:** Group the 71 features by category (numeric, demographic, diagnosis, medications) and show how many features belong to each group.

**Why:** Helps your team understand the composition of the feature matrix before modeling. Also useful for the presentation — shows you used a rich, multi-domain feature set.

In [ ]:
# Categorize all 71 features
def categorize_feature(col):
    if col in ['time_in_hospital','num_lab_procedures','num_procedures',
               'num_medications','number_outpatient','number_emergency',
               'number_inpatient','number_diagnoses']:
        return 'Numeric Clinical'
    if col in ['admission_type_id','discharge_disposition_id','admission_source_id']:
        return 'Admission / Discharge IDs'
    if col in ['gender_enc', 'age_enc']: return 'Demographics'
    if col.startswith('race_'):           return 'Demographics'
    if 'diag_' in col:                   return 'Diagnosis (ICD-9 groups)'
    if col.endswith('_enc') and col not in ['gender_enc','age_enc']: return 'Medications'
    if 'glu' in col or 'A1C' in col:     return 'Lab Results'
    return 'Other'

feature_groups = pd.Series({col: categorize_feature(col) for col in X_train.columns})
group_counts = feature_groups.value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Pie chart
colors_g = [C_RED, C_BLUE, C_GREEN, C_AMBER, C_GRAY, '#8B5CF6']
axes[0].pie(group_counts.values, labels=group_counts.index,
            colors=colors_g[:len(group_counts)], autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 10})
axes[0].set_title(f'Feature Composition ({X_train.shape[1]} Total Features)')

# Bar chart with counts
bars = axes[1].barh(group_counts.index, group_counts.values,
                    color=colors_g[:len(group_counts)], edgecolor='black', linewidth=0.5)
axes[1].set_title('Feature Count by Category')
axes[1].set_xlabel('Number of Features')
for bar, val in zip(bars, group_counts.values):
    axes[1].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                 f'{val} features', va='center', fontsize=10)

plt.suptitle('Model Feature Set Composition', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_12_feature_groups.png', dpi=150, bbox_inches='tight')
plt.show()

print('Feature breakdown:')
for group, count in group_counts.items():
    print(f'  {group:<30} {count:>3} features  ({count/X_train.shape[1]*100:.1f}%)')
print(f'  {"TOTAL":<30} {X_train.shape[1]:>3} features')

## CELL 15 — EDA Summary & Key Findings

**What:** Print a consolidated summary of the most important findings from this notebook.

**Why:** Gives you a ready-to-use bullet list for the Data Understanding section of your executive summary and presentation slides.

In [ ]:
# Calculate key stats for summary
pct_over60   = df[df['age'].isin(['[60-70)','[70-80)','[80-90)','[90-100)'])].shape[0] / len(df) * 100
rate_0prior  = df[df['number_inpatient']==0]['readmitted_30'].mean() * 100
rate_3plus   = df[df['number_inpatient']>=3]['readmitted_30'].mean() * 100
pct_caucasian = (df['race']=='Caucasian').mean() * 100

diag_rates = df.groupby('diag_1_cat')['readmitted_30'].mean().mul(100)
circ_rate  = diag_rates.get('Circulatory', 0)
resp_rate  = diag_rates.get('Respiratory', 0)

print('=' * 65)
print('  EDA KEY FINDINGS SUMMARY')
print('  HealthBridge Medical Center — 30-Day Readmission Prediction')
print('=' * 65)
print()
print('DATASET OVERVIEW')
print(f'  Total patient encounters (cleaned):    {len(df):,}')
print(f'  Positive class (readmitted <30 days):  {df["readmitted_30"].sum():,} ({df["readmitted_30"].mean()*100:.1f}%)')
print(f'  Class imbalance ratio:                 {(df["readmitted_30"]==0).sum()/(df["readmitted_30"]==1).sum():.1f}:1')
print(f'  Features after preprocessing:         {X_train.shape[1]}')
print()
print('DEMOGRAPHICS')
print(f'  Patients aged 60+:                    {pct_over60:.1f}% of total population')
print(f'  Race — Caucasian majority:             {pct_caucasian:.1f}%')
print(f'  Gender split:                          53.2% Female / 46.8% Male')
print()
print('KEY READMISSION PREDICTORS (from correlation analysis)')
print(f'  1. number_inpatient (prior visits):    STRONGEST — 0 visits: {rate_0prior:.1f}% vs 3+ visits: {rate_3plus:.1f}%')
print(f'  2. time_in_hospital:                   Longer stays → higher readmission risk')
print(f'  3. number_diagnoses:                   More diagnoses → more complex cases')
print(f'  4. num_medications:                    Polypharmacy correlates with readmission')
print(f'  5. discharge_disposition_id:           Where discharged to is a key predictor')
print()
print('DIAGNOSIS INSIGHTS (HRRP-relevant)')
print(f'  Circulatory (HF, AMI):                {circ_rate:.1f}% readmission rate')
print(f'  Respiratory (COPD, Pneumonia):         {resp_rate:.1f}% readmission rate')
print(f'  Overall average:                       {overall_rate:.1f}%')
print()
print('PREPROCESSING VALIDATION')
print(f'  SMOTE training class balance:          {y_train.mean()*100:.1f}% positive  ✅')
print(f'  Test set class balance (real world):   {y_test.mean()*100:.1f}% positive  ✅')
print()
print('NEXT STEP: Run Notebook 04 — Model Training')
print('  Load: X_train.xlsx, y_train.xlsx, X_test.xlsx, y_test.xlsx')
print('=' * 65)

---
## 🎯 EDA Complete

All charts have been saved to your working directory:

| File | Content |
|---|---|
| `eda_01_target_distribution.png` | Class imbalance — 3-class and binary |
| `eda_02_demographics.png` | Age, gender, race distributions |
| `eda_03_readmission_by_demographics.png` | Readmission rates by age and gender |
| `eda_04_prior_visits.png` | Prior inpatient visits vs readmission risk |
| `eda_05_feature_distributions.png` | Feature distributions by outcome |
| `eda_06_diagnosis_categories.png` | Readmission rate by ICD-9 category |
| `eda_07_medications.png` | Insulin, diabetes meds, A1C analysis |
| `eda_08_correlation.png` | Correlation heatmap |
| `eda_09_smote_validation.png` | Class balance before/after SMOTE |
| `eda_10_scaled_features.png` | Scaled feature distributions |
| `eda_11_feature_correlation.png` | Top/bottom correlated features |
| `eda_12_feature_groups.png` | Feature set composition |

---
*HealthBridge Medical Center | Advanced Data Analytics Capstone | AY 2025–2026*